In [6]:
import pandas as pd
import numpy as np
df = pd.read_csv("delivery_investigation.csv")

In [7]:
df.shape

(1278, 17)

In [8]:
df = df[df.order_id.duplicated() == False]

In [9]:
df = df[df.delivery_minutes > 0]

In [10]:
df = df[df.prep_minutes > 0]

In [11]:
df['distance_km'] = df['distance_km'].str.replace(',', '.').astype(float)

In [12]:
df['order_sum'] = df['order_sum'].str.replace(' ₽', '')
df['order_sum'] = df['order_sum'].str.replace(' ', '').astype(int)

In [13]:
import matplotlib.pyplot as plt

In [16]:
delivered_df = df[df['status'] == 'delivered'].copy()
delivered_df['is_late'] = delivered_df['delivery_minutes'] > delivered_df['promised_minutes']
point1_df = delivered_df.groupby('district')['is_late'].agg(
    Доля_опозданий='mean',
    Кол_во_опозданий='sum',
    Всего_заказов='count'
).reset_index()
print(point1_df.sort_values(by='Доля_опозданий', ascending=False))

   district  Доля_опозданий  Кол_во_опозданий  Всего_заказов
17      Юг         1.000000                 1              1
18      Юг         0.500000                 1              2
22       юг        0.500000                 1              2
16       Юг        0.475962                99            208
10   Север         0.333333                 1              3
21    центр        0.333333                 1              3
15       ЮГ        0.250000                 1              4
1    Восток        0.239521                40            167
4     Запад        0.215909                38            176
8     Север        0.212644                37            174
12    Центр        0.210909                58            275
0    ВОСТОК        0.000000                 0              3
5    Запад         0.000000                 0              3
3   Восток         0.000000                 0              3
2   Восток         0.000000                 0              1
6    Запад         0.000

In [18]:
point2_df = delivered_df.groupby(['district', 'transport'])['delivery_minutes'].agg(
    Среднее='mean',
    Медиана='median',
    Количество='count'
).reset_index()
print(point2_df)

   district transport    Среднее  Медиана  Количество
0    ВОСТОК      bike  34.300000    34.30           2
1    ВОСТОК   scooter  34.400000    34.40           1
2    Восток      BIKE  25.500000    30.00           3
3    Восток      bike  40.287097    40.25          62
4    Восток       car  33.647222    31.10          36
..      ...       ...        ...      ...         ...
66    север   scooter  34.900000    34.90           1
67    центр      bike  41.650000    41.65           2
68    центр       car  37.700000    37.70           1
69       юг      bike  53.000000    53.00           1
70       юг       car  28.200000    28.20           1

[71 rows x 5 columns]


In [20]:
delivered_df['delay_minutes'] = delivered_df['delivery_minutes'] - delivered_df['promised_minutes']
point3_weather = delivered_df.groupby('weather')['delay_minutes'].agg(
    Среднее_опоздание='mean',
    Медиана_опоздания='median'
).reset_index()
print(point3_weather.sort_values(by='Среднее_опоздание', ascending=False))

  weather  Среднее_опоздание  Медиана_опоздания
2    snow         170.834737               2.90
1    rain          43.960000              -6.20
0   clear          39.469146              -7.65


In [23]:
point3_load = delivered_df.groupby('courier_load')['delay_minutes'].agg(
    Среднее_опоздание='mean',
    Медиана_опоздания='median',
    Количество='count'
).reset_index()
print(point3_load)

   courier_load  Среднее_опоздание  Медиана_опоздания  Количество
0             1          66.482870               -7.8         432
1             2          50.857493               -6.0         367
2             3          36.452850               -6.0         193
3             4           2.742857               -0.4          49


In [24]:
point3_corr = delivered_df[['delay_minutes', 'courier_load', 'distance_km']].corr(method='spearman')

In [25]:
print(point3_corr)

               delay_minutes  courier_load  distance_km
delay_minutes       1.000000      0.131266     0.174102
courier_load        0.131266      1.000000     0.061957
distance_km         0.174102      0.061957     1.000000


In [26]:
df_raw = pd.read_csv('delivery_investigation.csv', encoding='utf-8-sig')